# AWG Initial Design Notebook

这个 notebook 用于把已经得到的 waveguide MODE 结果转成一版适合课程项目推进的 4-channel C-band AWG 初始设计参数。

本 notebook 不运行新的 Tidy3D 云端仿真，因此不会消耗新的 Tidy3D credit。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()

width_sweep_csv = PROJECT_ROOT / "waveguide_width_sweep_results.csv"
awg_params_csv = PROJECT_ROOT / "awg_initial_parameters.csv"
awg_channels_csv = PROJECT_ROOT / "awg_channel_plan.csv"
awg_lengths_csv = PROJECT_ROOT / "awg_arrayed_waveguide_lengths.csv"

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Load the MODE sweep results.
width_df = pd.read_csv(width_sweep_csv)
width_df

In [ ]:
# Choose the first-pass AWG design baseline.
selected_width_um = 0.50
selected_row = width_df[np.isclose(width_df["width_um"], selected_width_um)].iloc[0]

selected_neff = float(selected_row["n_eff_mode0"])
selected_ngroup = float(selected_row["n_group_mode0"])

print(f"Selected width = {selected_width_um:.2f} um")
print(f"Selected n_eff = {selected_neff:.6f}")
print(f"Selected n_group = {selected_ngroup:.6f}")

In [ ]:
# Define a practical 4-channel C-band target for the course project.
channel_count = 4
center_wavelength_nm = 1550.0
channel_spacing_nm = 1.6  # about 200 GHz near 1550 nm

channel_offsets = np.array([-1.5, -0.5, 0.5, 1.5]) * channel_spacing_nm
channel_centers_nm = center_wavelength_nm + channel_offsets

c = 299792458.0
channel_freq_thz = c / (channel_centers_nm * 1e-9) / 1e12

channel_df = pd.DataFrame(
    {
        "channel_id": np.arange(1, channel_count + 1),
        "center_wavelength_nm": channel_centers_nm,
        "center_frequency_THz": channel_freq_thz,
    }
)
channel_df

In [ ]:
# Estimate a first-pass AWG order and path length difference.
channel_center_span_nm = float(channel_centers_nm.max() - channel_centers_nm.min())
target_fsr_nm = 4.0 * channel_center_span_nm
awg_order_m = int(round(center_wavelength_nm / target_fsr_nm))
actual_fsr_nm = center_wavelength_nm / awg_order_m
delta_length_um = awg_order_m * (center_wavelength_nm / 1000.0) / selected_ngroup

print(f"Channel center span = {channel_center_span_nm:.4f} nm")
print(f"Target FSR = {target_fsr_nm:.4f} nm")
print(f"Chosen AWG order m = {awg_order_m}")
print(f"Actual FSR from lambda0 / m = {actual_fsr_nm:.4f} nm")
print(f"Estimated delta length = {delta_length_um:.4f} um")

In [ ]:
# Prepare a simple first-pass geometry parameter set.
arrayed_waveguide_count = 16
input_waveguide_count = 1
output_waveguide_count = 4

suggested_fpr_length_um = 50.0
suggested_array_pitch_um = 1.5
suggested_output_pitch_um = 2.0
minimum_bend_radius_um = 10.0

relative_lengths_um = np.arange(arrayed_waveguide_count) * delta_length_um
length_df = pd.DataFrame(
    {
        "arm_index": np.arange(arrayed_waveguide_count),
        "relative_length_um": relative_lengths_um,
    }
)
length_df.head()

In [ ]:
# Save the parameter tables for report writing and later modeling.
params_df = pd.DataFrame(
    [
        ["selected_waveguide_width", selected_width_um, "um", "Chosen from width sweep as the first AWG design baseline"],
        ["waveguide_height", 0.22, "um", "Fixed silicon strip waveguide thickness"],
        ["core_index", 3.48, "dimensionless", "Silicon refractive index used in MODE model"],
        ["cladding_index", 1.44, "dimensionless", "SiO2 refractive index used in MODE model"],
        ["selected_neff", selected_neff, "dimensionless", "Fundamental mode effective index at 1550 nm"],
        ["selected_ngroup", selected_ngroup, "dimensionless", "Fundamental mode group index at 1550 nm"],
        ["channel_count", channel_count, "count", "Simplified 4-channel demultiplexer target"],
        ["center_wavelength", center_wavelength_nm, "nm", "C-band design center"],
        ["channel_spacing", channel_spacing_nm, "nm", "Approximately 200 GHz near 1550 nm"],
        ["channel_center_span", channel_center_span_nm, "nm", "Span between the first and last channel centers"],
        ["target_fsr", target_fsr_nm, "nm", "Chosen as about 4x the 4-channel center span for safe separation"],
        ["awg_order_m", awg_order_m, "dimensionless", "Rounded from lambda0 / target_fsr"],
        ["actual_fsr", actual_fsr_nm, "nm", "Computed from lambda0 / m"],
        ["delta_length", delta_length_um, "um", "Arrayed waveguide path length difference from m*lambda0/ng"],
        ["arrayed_waveguide_count", arrayed_waveguide_count, "count", "Reasonable first-pass arm count for a simplified course project"],
        ["input_waveguide_count", input_waveguide_count, "count", "Single input demultiplexer configuration"],
        ["output_waveguide_count", output_waveguide_count, "count", "One output per channel"],
        ["suggested_fpr_length", suggested_fpr_length_um, "um", "Heuristic first-pass slab/FPR length for compact initial geometry"],
        ["suggested_array_pitch", suggested_array_pitch_um, "um", "Heuristic pitch at the array/FPR interface"],
        ["suggested_output_pitch", suggested_output_pitch_um, "um", "Heuristic output waveguide pitch before further tuning"],
        ["minimum_bend_radius", minimum_bend_radius_um, "um", "Conservative first-pass bend radius suggestion"],
    ],
    columns=["parameter", "value", "unit", "notes"],
)

params_df.to_csv(awg_params_csv, index=False)
channel_df.to_csv(awg_channels_csv, index=False)
length_df.to_csv(awg_lengths_csv, index=False)

print(f"Saved: {awg_params_csv}")
print(f"Saved: {awg_channels_csv}")
print(f"Saved: {awg_lengths_csv}")

In [ ]:
# Plot the channel centers.
plt.figure(figsize=(6, 4))
plt.stem(channel_df["center_wavelength_nm"], np.ones(len(channel_df)), basefmt=" ")
plt.xlabel("Channel center wavelength (nm)")
plt.ylabel("Marker")
plt.title("4-Channel C-Band Target Wavelengths")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot the relative arrayed waveguide lengths.
plt.figure(figsize=(7, 4))
plt.plot(length_df["arm_index"], length_df["relative_length_um"], marker="o")
plt.xlabel("Arrayed waveguide arm index")
plt.ylabel("Relative length (um)")
plt.title("Relative Arrayed Waveguide Lengths")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
params_df